In [6]:
from tensorflow import keras
import tensorflow as tf

from tensorflow.keras import models
from tensorflow.keras import layers

In [ ]:
class SimpleDense(keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.units = units
        self.activation = activation

    
    # build() method is called automatically the first time the 
    # layer is called (via its__call__() method).
    def build(self, input_shape):
        input_dim = input_shape[-1]
        self.W = self.add_weight(shape=(input_dim, self.units),
                                 initializer="random_normal")
        self.b = self.add_weight(shape=(self.units,),
                                 initializer="zeros")
        
    # forward pass
    def call(self, inputs):
        y = tf.matmul(inputs, self.W) + self.b
        if self.activation is not None:
            y = self.activation(y)
        return y

In [ ]:
# uses __call__()method
my_dense = SimpleDense(units=32, activation=tf.nn.relu)
input_tensor = tf.ones(shape=(2, 784))
output_tensor = my_dense(input_tensor)
print(output_tensor.shape)

(2, 32)


E0000 00:00:1774446812.354285   10263 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In `SimpleDense` we create weights in a dedicated state-creation method, `build()`, which receives as an argument the first input shape seen by the layer.

The `build()` method is called automatically the first time the layer is called, via its `__call__()` method.

In fact, that’s why we defined the computation in a separate `call()` method rather than in the `__call__()` method directly.

The `__call__()` method of the base layer schematically looks like this:

In [5]:
model = keras.Sequential([
    SimpleDense(32, activation="relu"),
    SimpleDense(64, activation="relu"),
    SimpleDense(32, activation="relu"),
    SimpleDense(10, activation="softmax")
])

Compile step

Loss function (objective function)—The quantity that will be minimized during
training. It represents a measure of success for the task at hand.

Optimizer—Determines how the network will be updated based on the loss func-
tion. It implements a specific variant of stochastic gradient descent (SGD)

Metrics—The measures of success you want to monitor during training and vali-
dation, such as classification accuracy. __Unlike the loss, training will not optimize
directly for these metrics__. As such, metrics don’t need to be differentiable.

In [ ]:
model = keras.Sequential([keras.layers.Dense(1)])
model.compile(optimizer="rmsprop",
              loss="mean_squared_error",
              metrics=["accuracy"])

#or with the object
model.compile(optimizer=keras.optimizers.RMSprop(),
              loss=keras.losses.MeanSquaredError(),
              metrics=[keras.metrics.BinaryAccuracy()])

# usefull for custom metrics or to pass arguments directly:
model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-4),
              loss=my_custom_loss,
              metrics=[my_custom_metric_1, my_custom_metric_2])

_IncompleteInputError: incomplete input (3476429588.py, line 11)

In [ ]:
history = model.fit(inputs,
                    targets,
                    epochs=5,
                    batch_size=128)

In [ ]:
>>> history.history
{"binary_accuracy": [0.855, 0.9565, 0.9555, 0.95, 0.951],
"loss": [0.6573270302042366,
0.07434618508815766,
0.07687718723714351,
0.07412414988875389,
0.07617757616937161]}


In [ ]:
model = keras.Sequential([keras.layers.Dense(1)])
model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.1),
              loss=keras.losses.MeanSquaredError(),
              metrics=[keras.metrics.BinaryAccuracy()])

# Shulle data
indices_permutation = np.random.permutation(len(inputs))
shuffled_inputs = inputs[indices_permutation]
shuffled_targets = targets[indices_permutation]
num_validation_samples = int(0.3 * len(inputs))
# define train and val data
val_inputs = shuffled_inputs[:num_validation_samples]
val_targets = shuffled_targets[:num_validation_samples]
training_inputs = shuffled_inputs[num_validation_samples:]
training_targets = shuffled_targets[num_validation_samples:]

# fit model
model.fit(training_inputs,
          training_targets,
          epochs=5,
          batch_size=16,
          Validation_data=(val_inputs, val_targets)

In [ ]:
##Note that if you want to compute the validation loss and metrics after the training
#is complete, you can call the evaluate() method:
loss_and_metrics = model.evaluate(val_inputs, val_targets, batch_size=128)